In [3]:
import pandas as pd

# قراءة ملف الاستهلاك - نتجاهل أول 4 أسطر معلومات إضافية
consumption = pd.read_csv('../data/raw/Retail_sales_of_electricity.csv', skiprows=4)

# قراءة ملف الطقس - نتجاهل الأسطر اللي بتبدأ بـ #
weather = pd.read_csv('../data/raw/weather_texas.csv', comment='#')

print("Consumption shape:", consumption.shape)
print("Weather shape:", weather.shape)

Consumption shape: (16, 309)
Weather shape: (307, 3)


In [4]:
# نعرض أول عمودين بس (description و units) لكل الصفوف
print(consumption[['description', 'units']])

                       description                  units
0      Retail sales of electricity  million kilowatthours
1                    United States  million kilowatthours
2      United States : all sectors  million kilowatthours
3      United States : residential  million kilowatthours
4       United States : commercial  million kilowatthours
5       United States : industrial  million kilowatthours
6   United States : transportation  million kilowatthours
7            United States : other  million kilowatthours
8               West South Central                    NaN
9                            Texas  million kilowatthours
10             Texas : all sectors  million kilowatthours
11             Texas : residential  million kilowatthours
12              Texas : commercial  million kilowatthours
13              Texas : industrial  million kilowatthours
14          Texas : transportation  million kilowatthours
15                   Texas : other  million kilowatthours


In [5]:
# سحب صف "Texas : all sectors" بس
texas_row = consumption[consumption['description'] == 'Texas : all sectors']

# حذف الأعمدة اللي مش شهور (description, units, source key)
texas_row = texas_row.drop(columns=['description', 'units', 'source key'])

# تحويل الشكل من عمود لكل شهر → صف لكل شهر
texas_long = texas_row.melt(var_name='month', value_name='consumption')

print(texas_long.head())
print(texas_long.shape)

      month consumption
0  Jan 2001       27584
1  Feb 2001       24171
2  Mar 2001       22424
3  Apr 2001       22635
4  May 2001       24516
(306, 2)


In [6]:
# تحويل عمود month لتاريخ حقيقي
texas_long['date'] = pd.to_datetime(texas_long['month'], format='%b %Y')

print(texas_long.head())
print(texas_long.dtypes)

      month consumption       date
0  Jan 2001       27584 2001-01-01
1  Feb 2001       24171 2001-02-01
2  Mar 2001       22424 2001-03-01
3  Apr 2001       22635 2001-04-01
4  May 2001       24516 2001-05-01
month                     str
consumption               str
date           datetime64[us]
dtype: object


In [7]:
# تحويل عمود consumption لأرقام حقيقية
texas_long['consumption'] = pd.to_numeric(texas_long['consumption'], errors='coerce')

print(texas_long.dtypes)
print(texas_long.isna().sum())

month                     str
consumption             int64
date           datetime64[us]
dtype: object
month          0
consumption    0
date           0
dtype: int64


In [8]:
# تحويل عمود Date لتاريخ حقيقي
weather['date'] = pd.to_datetime(weather['Date'], format='%Y%m')

print(weather.head())
print(weather.dtypes)

     Date  Value  Departure from Average       date
0  200101   43.9                    -3.1 2001-01-01
1  200102   51.9                     1.1 2001-02-01
2  200103   53.3                    -6.1 2001-03-01
3  200104   67.8                     1.5 2001-04-01
4  200105   74.1                     0.2 2001-05-01
Date                               int64
Value                            float64
Departure from Average           float64
date                      datetime64[us]
dtype: object


In [9]:
# دمج الملفين على أساس عمود date
final_df = pd.merge(texas_long, weather, on='date', how='inner')

print(final_df.head())
print(final_df.shape)

      month  consumption       date    Date  Value  Departure from Average
0  Jan 2001        27584 2001-01-01  200101   43.9                    -3.1
1  Feb 2001        24171 2001-02-01  200102   51.9                     1.1
2  Mar 2001        22424 2001-03-01  200103   53.3                    -6.1
3  Apr 2001        22635 2001-04-01  200104   67.8                     1.5
4  May 2001        24516 2001-05-01  200105   74.1                     0.2
(306, 6)


In [10]:
# حذف الأعمدة الزيادة
final_df = final_df.drop(columns=['month', 'Date'])

# إعادة تسمية بعض الأعمدة عشان تكون أوضح
final_df = final_df.rename(columns={
    'Value': 'avg_temp_f',
    'Departure from Average': 'temp_departure'
})

print(final_df.head())
print(final_df.columns.tolist())

   consumption       date  avg_temp_f  temp_departure
0        27584 2001-01-01        43.9            -3.1
1        24171 2001-02-01        51.9             1.1
2        22424 2001-03-01        53.3            -6.1
3        22635 2001-04-01        67.8             1.5
4        24516 2001-05-01        74.1             0.2
['consumption', 'date', 'avg_temp_f', 'temp_departure']


In [11]:
# حفظ الملف النهائي في data/processed
final_df.to_csv('../data/processed/texas_monthly_dataset.csv', index=False)

print("Completely Saved")

Completely Saved
